# COMEX Cash-and-Carry Arbitrage

Physical gold (GC) and silver (SI) calendar-spread carry strategy.

| Parameter | Value |
|-----------|-------|
| Metals | GC (all 12 months), SI (Jan/Mar/May/Jul/Sep/Dec) |
| Tenor | FDD1 → FDD2 (first business day of delivery month) |
| Carry | Switch − Funding (SOFR OIS, 15-pillar) − Storage |
| Direction | **LONG ONLY** |
| Signal | Rolling z-score of `carry_bp_ann` |
| Carry lock | Locked at entry — accrued daily throughout hold |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from itertools import product
from typing import Optional

# ── CONFIG ───────────────────────────────────────────────────────────────────
CONFIG = {
    # Data
    'DATA_PATH':         Path('data/data.csv'),
    'OUT_DIR':           Path('outputs_comex'),

    # Contract months
    'GC_MONTHS':         list(range(1, 13)),        # all 12 months
    'SI_MONTHS':         [1, 3, 5, 7, 9, 12],       # Jan Mar May Jul Sep Dec

    # Physical storage cost (bp/pa)
    'GC_STORAGE_BPPA':   15.0,
    'SI_STORAGE_BPPA':   25.0,

    # 15-pillar SOFR OIS curve  (column_name, tenor_days)
    'SOFR_PILLARS': [
        ('sofr_ois_1w',   7), ('sofr_ois_2w',  14),
        ('sofr_ois_1m',  30), ('sofr_ois_2m',  61), ('sofr_ois_3m',  91),
        ('sofr_ois_4m', 122), ('sofr_ois_5m', 152), ('sofr_ois_6m', 182),
        ('sofr_ois_7m', 213), ('sofr_ois_8m', 243), ('sofr_ois_9m', 273),
        ('sofr_ois_10m',304), ('sofr_ois_11m',334),
        ('sofr_ois_1y', 365), ('sofr_ois_2y', 730),
    ],

    # Signal
    'Z_WINDOW':          63,    # 3-month rolling window

    # Backtest defaults
    'ENTRY_Z':           1.5,
    'EXIT_Z':            0.3,
    'STOP_Z':            3.5,
    'EXEC_COST_BP':      0.50,  # per leg (1.0 bp round-trip)
    'MIN_DTE_ENTRY':     5,     # block entry within 5 days of FDD1
}
CONFIG['OUT_DIR'].mkdir(exist_ok=True)
print('CONFIG loaded.')
print(f"  SOFR pillars : {len(CONFIG['SOFR_PILLARS'])}")
print(f"  GC months    : {CONFIG['GC_MONTHS']}")
print(f"  SI months    : {CONFIG['SI_MONTHS']}")

## Section 1 — Data Load

Load `data/data.csv`, parse dates, validate required columns.

In [ ]:
df = pd.read_csv(CONFIG['DATA_PATH'], parse_dates=['date'], index_col='date')
df.sort_index(inplace=True)

# Validate key columns
REQ = ['gc_fut_front', 'gc_fut_second', 'si_fut_front', 'si_fut_second', 'fed_funds_target']
missing = [c for c in REQ if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns: {missing}')

print(f'Loaded {len(df):,} rows  {df.index[0].date()} → {df.index[-1].date()}')
print(f'Total columns: {len(df.columns)}')
print()

# Coverage audit
for col in ['gc_fut_front', 'gc_fut_second', 'si_fut_front', 'si_fut_second']:
    n = df[col].notna().sum()
    print(f'  {col:20s}  {n:5d} rows  ({n/len(df)*100:.0f}%  non-NaN)')

# SOFR pillar availability
available_pillars = [(c, d) for c, d in CONFIG['SOFR_PILLARS'] if c in df.columns]
print(f'\nSOFR pillars in data: {len(available_pillars)} / {len(CONFIG["SOFR_PILLARS"])}')
print('  ' + ', '.join(c for c, _ in available_pillars))

## Section 2 — FDD Calendar

Compute **First Delivery Day** for each contract month:
- **GC**: all 12 calendar months (COMEX gold delivers every month)
- **SI**: Jan / Mar / May / Jul / Sep / Dec (COMEX silver cycle)

**FDD = first business day of the delivery month** (Mon–Fri, weekday roll-forward only).  

For each trading date we find the *current front* and *second* FDD, then compute:
- `dte_fdd_gc1 / dte_fdd_si1` — calendar days to FDD of front contract
- `dte_fdd_gc2 / dte_fdd_si2` — calendar days to FDD of second contract
- `day_count_gc / day_count_si` — `dte2 − dte1` (physical hold period)

In [ ]:
def first_biz_day(year: int, month: int) -> pd.Timestamp:
    """Return first Mon-Fri day of given year/month (no holiday calendar)."""
    d = np.datetime64(f'{year:04d}-{month:02d}-01', 'D')
    return pd.Timestamp(np.busday_offset(d, 0, roll='forward'))


def build_fdd_list(months: list, years) -> list:
    """Sorted list of FDD timestamps for given months across year range."""
    return sorted(first_biz_day(y, m) for y in years for m in months)


def compute_dte_pair(
    dates: pd.DatetimeIndex,
    fdd_list: list,
) -> tuple:
    """
    For each date, find next FDD1 (>= date) and the subsequent FDD2.
    Returns (dte1_arr, dte2_arr) in calendar days.
    """
    fdd_ns = np.array([f.value for f in fdd_list], dtype='int64')
    date_ns = dates.astype('int64').values

    dte1 = np.full(len(dates), np.nan)
    dte2 = np.full(len(dates), np.nan)

    for i, (d, dv) in enumerate(zip(dates, date_ns)):
        idx = int(np.searchsorted(fdd_ns, dv, side='left'))
        if idx >= len(fdd_list) - 1:
            continue
        dte1[i] = (fdd_list[idx]     - d).days
        dte2[i] = (fdd_list[idx + 1] - d).days

    return dte1, dte2


# Build FDD lists spanning data range +/- 2 years
years = range(df.index[0].year - 1, df.index[-1].year + 3)
gc_fdds = build_fdd_list(CONFIG['GC_MONTHS'], years)
si_fdds = build_fdd_list(CONFIG['SI_MONTHS'], years)

gc_dte1, gc_dte2 = compute_dte_pair(df.index, gc_fdds)
si_dte1, si_dte2 = compute_dte_pair(df.index, si_fdds)

df['dte_fdd_gc1'] = gc_dte1
df['dte_fdd_gc2'] = gc_dte2
df['day_count_gc'] = df['dte_fdd_gc2'] - df['dte_fdd_gc1']

df['dte_fdd_si1'] = si_dte1
df['dte_fdd_si2'] = si_dte2
df['day_count_si'] = df['dte_fdd_si2'] - df['dte_fdd_si1']

print('FDD calendar computed.')
n_gc_in_range = sum(1 for f in gc_fdds if df.index[0] <= f <= df.index[-1])
n_si_in_range = sum(1 for f in si_fdds if df.index[0] <= f <= df.index[-1])
print(f'  GC FDDs in data range: {n_gc_in_range}')
print(f'  SI FDDs in data range: {n_si_in_range}')
print()

cols = ['dte_fdd_gc1', 'dte_fdd_gc2', 'day_count_gc',
        'dte_fdd_si1', 'dte_fdd_si2', 'day_count_si']
sample = df[cols].dropna()
print(f'Valid rows: {len(sample)}')
print(sample.describe().round(1))

## Section 3 — SOFR OIS Curve Interpolation (15 Pillars)

Build a daily SOFR OIS rate interpolated to the **physical hold tenor** (`day_count = dte_fdd2 − dte_fdd1`).

| Method | Detail |
|--------|--------|
| Interpolation | `numpy.interp` — piecewise linear between active pillars |
| Extrapolation | Flat (hold first/last pillar value) |
| Fallback | `fed_funds_target` when fewer than 2 pillars have data |
| Pillars | 1W 2W 1M 2M 3M 4M 5M 6M 7M 8M 9M 10M 11M 1Y 2Y |

In [ ]:
# Build active pillar arrays (columns present in data)
PILLAR_COLS = [c for c, _ in CONFIG['SOFR_PILLARS'] if c in df.columns]
PILLAR_DAYS = np.array([d for c, d in CONFIG['SOFR_PILLARS'] if c in df.columns])
print(f'Active SOFR pillars: {len(PILLAR_COLS)}')
for col, days in zip(PILLAR_COLS, PILLAR_DAYS):
    cov = df[col].notna().sum()
    print(f'  {col:20s}  {days:4d}d   {cov:5d} rows  ({cov/len(df)*100:.0f}%)')


def sofr_interp_vec(
    sofr_mat: np.ndarray,       # (N, n_pillars)
    pillar_days: np.ndarray,    # (n_pillars,)
    tenor_arr: np.ndarray,      # (N,)
    fallback_arr: np.ndarray,   # (N,)
) -> np.ndarray:
    """
    Row-by-row linear SOFR interpolation using numpy.interp.
    numpy.interp gives flat extrapolation at endpoints.
    Falls back to fed_funds_target when < 2 valid pillars.
    """
    results = np.full(len(tenor_arr), np.nan)
    for i in range(len(tenor_arr)):
        row = sofr_mat[i]
        valid = ~np.isnan(row)
        if valid.sum() < 2:
            results[i] = fallback_arr[i]  # fed_funds_target fallback
            continue
        xs = pillar_days[valid]
        ys = row[valid]
        tenor = tenor_arr[i]
        if np.isnan(tenor):
            continue
        results[i] = float(np.interp(tenor, xs, ys))
    return results


sofr_mat     = df[PILLAR_COLS].values
fallback_arr = df['fed_funds_target'].values if 'fed_funds_target' in df.columns \
               else np.full(len(df), np.nan)

df['sofr_interp_gc'] = sofr_interp_vec(
    sofr_mat, PILLAR_DAYS,
    df['day_count_gc'].values, fallback_arr
)
df.loc[df['day_count_gc'].isna(), 'sofr_interp_gc'] = np.nan

df['sofr_interp_si'] = sofr_interp_vec(
    sofr_mat, PILLAR_DAYS,
    df['day_count_si'].values, fallback_arr
)
df.loc[df['day_count_si'].isna(), 'sofr_interp_si'] = np.nan

print('\nSOFR interpolation complete.')
for col in ['sofr_interp_gc', 'sofr_interp_si']:
    s = df[col].dropna()
    print(f'  {col}:  mean={s.mean():.2f}%  min={s.min():.2f}%  max={s.max():.2f}%  n={len(s)}')

## Section 4 — Carry Calculation

**Cash-and-carry carry decomposition** (all in USD, then annualised to bp):

| Component | Formula |
|-----------|---------|
| Switch | `F2 − F1` (USD/oz) |
| Funding | `(SOFR_interp / 100) × t × F1` |
| Storage | `(storage_bppa / 10000) × t × F1` |
| **Net Carry** | `Switch − Funding − Storage` (USD) |
| Carry bp ann | `carry_usd / F1 / t × 10 000` |

Where `t = day_count / 360`.  
Positive carry ⟹ futures premium exceeds financing + storage cost.

In [ ]:
def compute_carry(df: pd.DataFrame, metal: str, storage_bppa: float) -> pd.DataFrame:
    """
    Compute carry decomposition for a metal.
    Adds switch_bp_ann, funding_bp_ann, storage_bp_ann, carry_bp_ann columns.
    """
    f1   = df[f'{metal}_fut_front'].astype(float)
    f2   = df[f'{metal}_fut_second'].astype(float)
    sofr = df[f'sofr_interp_{metal}'].astype(float)
    dc   = df[f'day_count_{metal}'].astype(float)

    t            = dc / 360.0
    switch_usd   = f2 - f1
    funding_usd  = (sofr / 100.0) * t * f1
    storage_usd  = (storage_bppa / 10000.0) * t * f1
    carry_usd    = switch_usd - funding_usd - storage_usd

    with np.errstate(divide='ignore', invalid='ignore'):
        scale = np.where((t > 0) & f1.notna() & (f1 > 0), 10000.0 / (f1 * t), np.nan)

    df[f'switch_usd_{metal}']    = switch_usd
    df[f'carry_usd_{metal}']     = carry_usd
    df[f'switch_bp_ann_{metal}'] = switch_usd   * scale
    df[f'funding_bp_ann_{metal}']= funding_usd  * scale
    df[f'storage_bp_ann_{metal}']= storage_usd  * scale
    df[f'carry_bp_ann_{metal}']  = carry_usd    * scale
    return df


df = compute_carry(df, 'gc', CONFIG['GC_STORAGE_BPPA'])
df = compute_carry(df, 'si', CONFIG['SI_STORAGE_BPPA'])

print('Carry computed.')
for metal, label in [('gc', 'Gold GC'), ('si', 'Silver SI')]:
    sw  = df[f'switch_bp_ann_{metal}'].dropna()
    fu  = df[f'funding_bp_ann_{metal}'].dropna()
    st  = df[f'storage_bp_ann_{metal}'].dropna()
    net = df[f'carry_bp_ann_{metal}'].dropna()
    print(f'\n{label}:')
    print(f'  Switch   : {sw.mean():+7.1f} bp  ({sw.min():.0f} – {sw.max():.0f})')
    print(f'  Funding  : {fu.mean():+7.1f} bp')
    print(f'  Storage  : {st.mean():+7.1f} bp')
    print(f'  Net carry: {net.mean():+7.1f} bp  ({net.min():.0f} – {net.max():.0f})')
    print(f'  % carry>0: {(net > 0).mean()*100:.0f}%')

## Section 5 — Z-Score Signal

Rolling z-score of `carry_bp_ann` (window = **63 trading days** ≈ 3 months).

```
z = (carry_bp_ann − rolling_mean) / rolling_std
```

**LONG** signal when `z ≥ entry_z` — carry is unusually wide relative to its recent history.

In [ ]:
W = CONFIG['Z_WINDOW']

for metal in ('gc', 'si'):
    c  = df[f'carry_bp_ann_{metal}']
    mu = c.rolling(W, min_periods=W // 2).mean()
    sd = c.rolling(W, min_periods=W // 2).std()
    df[f'carry_z_{metal}'] = (c - mu) / sd

print(f'Z-score computed (window = {W} days).')
for metal in ('gc', 'si'):
    s = df[f'carry_z_{metal}'].dropna()
    entry_pct  = (s >= CONFIG['ENTRY_Z']).mean() * 100
    stop_pct   = (s <= -CONFIG['STOP_Z']).mean() * 100
    print(f'  {metal.upper()}:  mean={s.mean():.2f}  std={s.std():.2f}  '
          f'pct≥+{CONFIG["ENTRY_Z"]}σ: {entry_pct:.0f}%  '
          f'pct≤-{CONFIG["STOP_Z"]}σ: {stop_pct:.0f}%')

## Section 6 — Carry History

Four-panel chart:
1. GC `carry_bp_ann` with rolling mean ± 1σ band
2. SI `carry_bp_ann` with rolling mean ± 1σ band
3. GC carry z-score with entry/exit/stop thresholds
4. SI carry z-score with entry/exit/stop thresholds

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('COMEX Cash-and-Carry — Carry History', fontsize=14, fontweight='bold')

EZ = CONFIG['ENTRY_Z']
XZ = CONFIG['EXIT_Z']
SZ = CONFIG['STOP_Z']

for col, (metal, label) in enumerate([('gc', 'Gold (GC)'), ('si', 'Silver (SI)')]):
    carry = df[f'carry_bp_ann_{metal}'].dropna()
    z     = df[f'carry_z_{metal}'].dropna()
    mu    = carry.rolling(W, min_periods=W // 2).mean()
    sd    = carry.rolling(W, min_periods=W // 2).std()

    # ── Row 0: carry level ──
    ax = axes[0, col]
    ax.fill_between(carry.index, mu - sd, mu + sd, alpha=0.15, color='steelblue', label='±1σ')
    ax.plot(carry.index, carry, lw=0.8, color='steelblue', alpha=0.9, label='carry bp ann')
    ax.plot(mu.index, mu, lw=1.4, color='navy', label='rolling mean')
    ax.axhline(0, color='black', lw=0.5, ls='--')
    ax.set_title(f'{label} — Carry (bp ann)', fontsize=11)
    ax.set_ylabel('bp ann')
    ax.legend(fontsize=8, loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.grid(alpha=0.3)

    # ── Row 1: z-score ──
    ax = axes[1, col]
    ax.plot(z.index, z, lw=0.8, color='darkorange', alpha=0.9)
    ax.axhline( EZ, color='green', lw=1.2, ls='--', label=f'entry +{EZ}σ')
    ax.axhline( XZ, color='blue',  lw=1.0, ls=':',  label=f'exit  +{XZ}σ')
    ax.axhline(-SZ, color='red',   lw=1.2, ls='--', label=f'stop  −{SZ}σ')
    ax.axhline(0,   color='black', lw=0.5, ls='--')
    ax.set_title(f'{label} — Carry Z-Score', fontsize=11)
    ax.set_ylabel('σ')
    ax.legend(fontsize=8, loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.grid(alpha=0.3)

plt.tight_layout()
out = CONFIG['OUT_DIR'] / 'cc_01_carry_history.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## Section 7 — Carry Attribution Decomposition

Stacked area chart showing how **Switch**, **−Funding**, and **−Storage** each contribute to net carry (bp ann) over time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Carry Attribution: Switch vs Funding vs Storage', fontsize=13, fontweight='bold')

for ax, (metal, label) in zip(axes, [('gc', 'Gold (GC)'), ('si', 'Silver (SI)')]):
    cols = [f'switch_bp_ann_{metal}', f'funding_bp_ann_{metal}',
            f'storage_bp_ann_{metal}', f'carry_bp_ann_{metal}']
    sub = df[cols].dropna()

    ax.stackplot(
        sub.index,
        sub[f'switch_bp_ann_{metal}'],
        -sub[f'funding_bp_ann_{metal}'],
        -sub[f'storage_bp_ann_{metal}'],
        labels=['Switch', '−Funding', '−Storage'],
        colors=['#2196F3', '#F44336', '#FF9800'],
        alpha=0.7,
    )
    ax.plot(sub.index, sub[f'carry_bp_ann_{metal}'],
            color='black', lw=1.4, label='Net Carry', zorder=5)
    ax.axhline(0, color='black', lw=0.5, ls='--')
    ax.set_title(label, fontsize=11)
    ax.set_ylabel('bp ann')
    ax.legend(fontsize=8, loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.grid(alpha=0.3)

plt.tight_layout()
out = CONFIG['OUT_DIR'] / 'cc_02_attribution_decomp.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## Section 8 — Backtest: LONG-Only Cash-and-Carry

**Strategy rules:**
- **Entry**: `carry_z ≥ entry_z`  AND  `dte_fdd1 > MIN_DTE_ENTRY` (block near-FDD entry)
- **Exit (signal)**: `carry_z ≤ exit_z` (carry has normalised)
- **Exit (stop)**: `carry_z ≤ −stop_z` (carry collapsed / reversed)
- **Exit (physical)**: `dte_fdd1 ≤ 0` (FDD1 reached — mandatory settlement)
- **Carry locked** at entry: accrues `entry_carry_bp_ann / 360` bp per day

**P&L decomposition:**
```
actual_pnl_bp  = (switch_entry − switch_exit) / F1_entry × 10 000
carry_pnl_bp   = entry_carry_bp_ann / 360 × hold_days          # theoretical accrual
basis_pnl_bp   = actual_pnl_bp − carry_pnl_bp                  # basis risk residual
total_pnl_bp   = actual_pnl_bp − exec_cost_bp × 2
               = basis_pnl_bp + carry_pnl_bp − exec_cost_bp × 2
```

**Trade log fields** (no underscores): `entrydate, exitdate, metal, direction, entrycarry, exitcarry, entryz, exitz_val, exitreason, holddays, entrydtefdd, basispnlbp, carrypnlbp, execcostbp_total, totalpnlbp`

In [ ]:
def backtest_cc(
    df: pd.DataFrame,
    metal: str = 'gc',
    entryz: float = 1.5,
    exitz: float = 0.3,
    stopz: float = 3.5,
    execcostbp: float = 0.50,
    iscutoff: Optional[pd.Timestamp] = None,
) -> tuple:
    """
    LONG-only cash-and-carry backtest.

    Parameters
    ----------
    df         : DataFrame with carry, z-score, FDD DTE, and futures price columns.
    metal      : 'gc' or 'si'.
    entryz     : Enter when carry_z >= entryz.
    exitz      : Exit (signal) when carry_z <= exitz.
    stopz      : Stop-loss when carry_z <= -stopz.
    execcostbp : Execution cost per leg (bp). Round-trip = 2×.
    iscutoff   : If set, ignore data after this date (IS/OOS split).

    Returns
    -------
    daily_pnl : pd.Series  — daily P&L in bp indexed by date.
    trade_log : pd.DataFrame — one row per completed trade.
    """
    f1_col    = f'{metal}_fut_front'
    f2_col    = f'{metal}_fut_second'
    carry_col = f'carry_bp_ann_{metal}'
    z_col     = f'carry_z_{metal}'
    dte_col   = f'dte_fdd_{metal}1'

    data = df.copy()
    if iscutoff is not None:
        data = data[data.index <= pd.Timestamp(iscutoff)]

    daily_pnl = pd.Series(0.0, index=data.index, dtype=float)
    trades: list = []

    in_trade     = False
    entry_date   = None
    entry_f1     = None
    entry_switch = None
    entry_carry  = None
    entry_zval   = None
    entry_dte    = None

    min_dte = CONFIG['MIN_DTE_ENTRY']

    for date, row in data.iterrows():
        z      = row[z_col]
        carry  = row[carry_col]
        f1     = row[f1_col]
        f2     = row[f2_col]
        dte1   = row[dte_col]

        # Skip rows with missing required values
        if any(pd.isna(v) for v in [z, carry, f1, f2, dte1]):
            continue

        if in_trade:
            # ── Daily carry accrual (locked at entry) ──
            daily_pnl.loc[date] += entry_carry / 360.0

            # ── Check exit conditions ──
            exit_trade  = False
            exit_reason = ''

            if dte1 <= 0:
                exit_trade  = True
                exit_reason = 'fdd_exit'
            elif z <= exitz:
                exit_trade  = True
                exit_reason = 'signal'
            elif z <= -stopz:
                exit_trade  = True
                exit_reason = 'stop'

            if exit_trade:
                exit_switch = f2 - f1
                # Long C&C: bought F1 / sold F2 at entry → profit when spread narrows
                actual_pnl  = (entry_switch - exit_switch) / entry_f1 * 10000.0
                hold_days   = (date - entry_date).days
                carry_pnl   = entry_carry / 360.0 * hold_days   # theoretical accrual
                basis_pnl   = actual_pnl - carry_pnl            # residual / basis risk
                total_pnl   = actual_pnl - execcostbp * 2.0

                # Book basis + exit exec on exit day (carry already in daily loop)
                daily_pnl.loc[date] += basis_pnl - execcostbp

                trades.append({
                    'entrydate':        entry_date,
                    'exitdate':         date,
                    'metal':            metal.upper(),
                    'direction':        'long',
                    'entrycarry':       round(entry_carry,  2),
                    'exitcarry':        round(carry,        2),
                    'entryz':           round(entry_zval,   2),
                    'exitz_val':        round(z,            2),
                    'exitreason':       exit_reason,
                    'holddays':         hold_days,
                    'entrydtefdd':      int(entry_dte),
                    'basispnlbp':       round(basis_pnl,    2),
                    'carrypnlbp':       round(carry_pnl,    2),
                    'execcostbp_total': round(execcostbp * 2.0, 2),
                    'totalpnlbp':       round(total_pnl,    2),
                })
                in_trade = False

        # ── Check entry (after potential exit on same day) ──
        if not in_trade and z >= entryz and dte1 > min_dte:
            in_trade     = True
            entry_date   = date
            entry_f1     = f1
            entry_switch = f2 - f1
            entry_carry  = carry
            entry_zval   = z
            entry_dte    = dte1
            # Deduct entry execution cost
            daily_pnl.loc[date] -= execcostbp

    trade_log = pd.DataFrame(trades)
    return daily_pnl, trade_log

In [ ]:
# ── Run backtest with default CONFIG params ──
daily_pnl_gc, tlog_gc = backtest_cc(
    df, metal='gc',
    entryz=CONFIG['ENTRY_Z'], exitz=CONFIG['EXIT_Z'],
    stopz=CONFIG['STOP_Z'],   execcostbp=CONFIG['EXEC_COST_BP'],
)
daily_pnl_si, tlog_si = backtest_cc(
    df, metal='si',
    entryz=CONFIG['ENTRY_Z'], exitz=CONFIG['EXIT_Z'],
    stopz=CONFIG['STOP_Z'],   execcostbp=CONFIG['EXEC_COST_BP'],
)


def summary_stats(pnl: pd.Series, label: str) -> None:
    cum = pnl.cumsum()
    ar  = pnl.mean() * 252
    vol = pnl.std()  * np.sqrt(252)
    sr  = ar / vol if vol > 0 else np.nan
    dd  = (cum - cum.cummax()).min()
    active_days = (pnl != 0).sum()
    print(f'\n{label}')
    print(f'  Ann return  : {ar:+7.1f} bp/yr')
    print(f'  Sharpe      :  {sr:.2f}')
    print(f'  Max drawdown: {dd:7.1f} bp')
    print(f'  Active days : {active_days}')
    print(f'  Total P&L   : {pnl.sum():+7.1f} bp')


summary_stats(daily_pnl_gc, 'GC (Gold C&C)')
summary_stats(daily_pnl_si, 'SI (Silver C&C)')

# ── Trade logs ──
SHOW_COLS = ['entrydate', 'exitdate', 'exitreason', 'holddays',
             'entrycarry', 'entrydtefdd', 'carrypnlbp', 'basispnlbp', 'totalpnlbp']

print('\n── GC Trade Log ──')
if len(tlog_gc):
    display(tlog_gc[SHOW_COLS])
print(f'Total GC  : {tlog_gc["totalpnlbp"].sum():.1f} bp  ({len(tlog_gc)} trades)')

print('\n── SI Trade Log ──')
if len(tlog_si):
    display(tlog_si[SHOW_COLS])
print(f'Total SI  : {tlog_si["totalpnlbp"].sum():.1f} bp  ({len(tlog_si)} trades)')

# Save
tlog_gc.to_csv(CONFIG['OUT_DIR'] / 'cc_trades_gc.csv', index=False)
tlog_si.to_csv(CONFIG['OUT_DIR'] / 'cc_trades_si.csv', index=False)
print('\nTrade logs saved.')

## Section 9 — Parameter Sweep

Grid search over `entry_z × exit_z × stop_z` (27 combinations × 2 metals = 54 backtests).  
Metrics: total P&L, annualised Sharpe, max drawdown, trade count.  
Results visualised as a Sharpe heatmap (`entry_z × exit_z` at `stop_z = 3.5`).

In [ ]:
SWEEP = {
    'entryz': [1.0, 1.5, 2.0],
    'exitz':  [0.0, 0.3, 0.5],
    'stopz':  [2.5, 3.5, 4.5],
}


def quick_stats(pnl: pd.Series) -> dict:
    cum = pnl.cumsum()
    ar  = pnl.mean() * 252
    vol = pnl.std()  * np.sqrt(252)
    sr  = ar / vol if vol > 0 else np.nan
    dd  = (cum - cum.cummax()).min()
    return {'ann_ret': ar, 'sharpe': sr, 'max_dd': dd}


combos = list(product(SWEEP['entryz'], SWEEP['exitz'], SWEEP['stopz']))
print(f'Running {len(combos) * 2} backtests ({len(combos)} combos × 2 metals)...')

rows = []
for ez, xz, sz in combos:
    for metal in ('gc', 'si'):
        pnl, tlog = backtest_cc(
            df, metal=metal, entryz=ez, exitz=xz, stopz=sz,
            execcostbp=CONFIG['EXEC_COST_BP'],
        )
        st = quick_stats(pnl)
        rows.append({
            'metal':     metal.upper(),
            'entryz':    ez,
            'exitz':     xz,
            'stopz':     sz,
            'total_pnl': round(pnl.sum(),     1),
            'ann_ret':   round(st['ann_ret'], 1),
            'sharpe':    round(st['sharpe'],  2),
            'max_dd':    round(st['max_dd'],  1),
            'n_trades':  len(tlog),
        })

sweep_df = pd.DataFrame(rows)
sweep_df.to_csv(CONFIG['OUT_DIR'] / 'cc_sweep_results.csv', index=False)

print('\nTop 10 by Sharpe — GC:')
display(sweep_df[sweep_df.metal == 'GC'].sort_values('sharpe', ascending=False).head(10))
print('\nTop 10 by Sharpe — SI:')
display(sweep_df[sweep_df.metal == 'SI'].sort_values('sharpe', ascending=False).head(10))

# ── Heatmap: Sharpe by entry_z × exit_z at fixed stop_z = 3.5 ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Sharpe Heatmap: entry_z × exit_z  (stop_z = 3.5)', fontsize=12)

for ax, metal in zip(axes, ['GC', 'SI']):
    sub = sweep_df[(sweep_df.metal == metal) & (sweep_df.stopz == 3.5)]
    pivot = sub.pivot(index='entryz', columns='exitz', values='sharpe')
    im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=-0.5, vmax=1.5)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_yticks(range(len(pivot.index)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel('exit_z')
    ax.set_ylabel('entry_z')
    ax.set_title(f'{metal} — Sharpe')
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            v = pivot.values[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=10,
                    color='black' if abs(v) < 1.0 else 'white')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
out = CONFIG['OUT_DIR'] / 'cc_03_sweep_heatmap.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## Section 10 — P&L Attribution per Trade

Decompose total P&L per trade into three components:

| Component | Description |
|-----------|-------------|
| **Carry P&L** | Locked carry accrual over hold period (`entry_carry / 360 × hold_days`) |
| **Basis P&L** | Actual spread move minus theoretical carry — measures basis convergence / divergence |
| **Exec Cost** | Round-trip commission (2 × exec_cost_bp) |

`total_pnl = carry_pnl + basis_pnl − exec_cost`

In [ ]:
tlog_all = pd.concat(
    [tlog_gc.assign(metal='GC'), tlog_si.assign(metal='SI')],
    ignore_index=True,
)

if tlog_all.empty:
    print('No trades to attribute.')
else:
    # ── Summary table ──
    summary = (
        tlog_all
        .groupby(['metal', 'exitreason'])
        [['basispnlbp', 'carrypnlbp', 'totalpnlbp', 'holddays']]
        .agg(count=('totalpnlbp', 'count'),
             mean_carry=('carrypnlbp', 'mean'),
             mean_basis=('basispnlbp', 'mean'),
             mean_total=('totalpnlbp', 'mean'),
             sum_total =('totalpnlbp', 'sum'),
             mean_hold =('holddays',   'mean'))
        .round(2)
    )
    print('Attribution summary by metal × exit reason:')
    display(summary)

    # ── Bar chart: per-trade attribution ──
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('P&L Attribution per Trade', fontsize=13)

    CMAP = {'carrypnlbp': '#4CAF50', 'basispnlbp': '#2196F3', 'execcostbp_total': '#F44336'}
    LABS = {'carrypnlbp': 'Carry P&L', 'basispnlbp': 'Basis P&L', 'execcostbp_total': 'Exec Cost'}

    for ax, metal in zip(axes, ['GC', 'SI']):
        sub = tlog_all[tlog_all.metal == metal].reset_index(drop=True)
        if sub.empty:
            ax.set_title(f'{metal} — No trades')
            continue

        x = np.arange(len(sub))
        w = 0.25
        for i, col in enumerate(['carrypnlbp', 'basispnlbp', 'execcostbp_total']):
            vals = sub[col] * (-1 if col == 'execcostbp_total' else 1)
            ax.bar(x + i * w, vals, w, label=LABS[col], color=CMAP[col], alpha=0.8)

        ax.plot(x + w, sub['totalpnlbp'], 'ko-', ms=4, lw=1, label='Total P&L', zorder=5)
        ax.axhline(0, color='black', lw=0.5)
        ax.set_title(f'{metal} — {len(sub)} trades')
        ax.set_xlabel('Trade #')
        ax.set_ylabel('bp')
        ax.legend(fontsize=8)
        ax.set_xticks(x + w)
        ax.set_xticklabels([f'T{i+1}' for i in range(len(sub))], fontsize=7)
        ax.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    out = CONFIG['OUT_DIR'] / 'cc_04_attribution.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

    tlog_all.to_csv(CONFIG['OUT_DIR'] / 'cc_attribution.csv', index=False)

## Section 11 — Cumulative P&L & Rolling Sharpe

Combined GC + SI equal-weight daily P&L stream, cumulative curve, and 63-day rolling Sharpe.

In [ ]:
pnl_combined = (daily_pnl_gc + daily_pnl_si).fillna(0)
cum_gc       = daily_pnl_gc.cumsum()
cum_si       = daily_pnl_si.cumsum()
cum_all      = pnl_combined.cumsum()

ROLL = 63
roll_sharpe = (
    pnl_combined.rolling(ROLL).mean() /
    pnl_combined.rolling(ROLL).std()
) * np.sqrt(252)

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(14, 9), sharex=True,
    gridspec_kw={'height_ratios': [2, 1]},
)
fig.suptitle('COMEX C&C — Cumulative P&L & Rolling Sharpe', fontsize=13, fontweight='bold')

ax1.plot(cum_gc.index,  cum_gc.values,  lw=1.2, color='#1565C0', label='GC',       alpha=0.8)
ax1.plot(cum_si.index,  cum_si.values,  lw=1.2, color='#BF360C', label='SI',       alpha=0.8)
ax1.plot(cum_all.index, cum_all.values, lw=2.0, color='black',   label='Combined', zorder=5)
ax1.fill_between(cum_all.index, cum_all, 0,
                 where=(cum_all >= 0), alpha=0.08, color='green')
ax1.fill_between(cum_all.index, cum_all, 0,
                 where=(cum_all  < 0), alpha=0.08, color='red')
ax1.axhline(0, color='grey', lw=0.5, ls='--')
ax1.set_ylabel('Cumulative P&L (bp)')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(alpha=0.3)

ax2.plot(roll_sharpe.index, roll_sharpe.values, lw=1.0, color='purple',
         label=f'{ROLL}d Rolling Sharpe')
ax2.axhline( 0, color='grey',  lw=0.5, ls='--')
ax2.axhline( 1, color='green', lw=0.8, ls=':')
ax2.axhline(-1, color='red',   lw=0.8, ls=':')
ax2.set_ylabel('Rolling Sharpe')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
out = CONFIG['OUT_DIR'] / 'cc_05_cum_pnl.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

print('\nFull-period statistics:')
for lbl, pnl in [('GC', daily_pnl_gc), ('SI', daily_pnl_si), ('Combined', pnl_combined)]:
    cum = pnl.cumsum()
    ar  = pnl.mean() * 252
    vol = pnl.std()  * np.sqrt(252)
    sr  = ar / vol if vol > 0 else np.nan
    dd  = (cum - cum.cummax()).min()
    print(f'  {lbl:10s}  ann={ar:+6.1f}bp  vol={vol:5.1f}  Sharpe={sr:5.2f}  MaxDD={dd:7.1f}bp')

## Section 12 — Stress Test

Evaluate strategy performance during four stress windows within the data range:

| Window | Dates | Event |
|--------|-------|-------|
| COVID Crash | 2020-02-20 → 2020-04-30 | Basis blow-out, repo stress |
| Fed Hike Cycle | 2022-01-01 → 2022-12-31 | Fastest rate hike since 1980s |
| Banking Crisis | 2023-03-01 → 2023-06-30 | SVB / Credit Suisse stress |
| XAG Lease Blowout | 2025-10-15 → 2025-11-30 | Silver-specific lease spike |

For each window: per-metal P&L (bp), max drawdown (bp), trade count.

In [ ]:
STRESS_WINDOWS = [
    ('COVID Crash',        '2020-02-20', '2020-04-30'),
    ('Fed Hike Cycle',     '2022-01-01', '2022-12-31'),
    ('Banking Crisis',     '2023-03-01', '2023-06-30'),
    ('XAG Lease Blowout',  '2025-10-15', '2025-11-30'),
]

stress_rows = []
for win_name, start, end in STRESS_WINDOWS:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    for metal in ('gc', 'si'):
        pnl, tlog = backtest_cc(
            df, metal=metal,
            entryz=CONFIG['ENTRY_Z'], exitz=CONFIG['EXIT_Z'],
            stopz=CONFIG['STOP_Z'],   execcostbp=CONFIG['EXEC_COST_BP'],
        )
        wpnl  = pnl.loc[s:e]
        cum_w = wpnl.cumsum()
        total = wpnl.sum()
        maxdd = (cum_w - cum_w.cummax()).min() if len(cum_w) > 0 else np.nan
        if len(tlog):
            n_tr = len(tlog[(tlog['entrydate'] >= s) & (tlog['entrydate'] <= e)])
        else:
            n_tr = 0
        stress_rows.append({
            'window':       win_name,
            'metal':        metal.upper(),
            'total_pnl_bp': round(total, 1),
            'max_dd_bp':    round(maxdd, 1),
            'n_trades':     n_tr,
        })

stress_df = pd.DataFrame(stress_rows)
print('Stress Test Results:')
pivot_stress = stress_df.pivot_table(
    index='window', columns='metal',
    values=['total_pnl_bp', 'max_dd_bp', 'n_trades'],
    aggfunc='first',
).round(1)
display(pivot_stress)

# ── Cumulative P&L within each stress window ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Stress Windows — Cumulative P&L', fontsize=13, fontweight='bold')

for ax, (win_name, start, end) in zip(axes.flatten(), STRESS_WINDOWS):
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    for metal, color, lbl in [('gc', '#1565C0', 'GC'), ('si', '#BF360C', 'SI')]:
        pnl, _ = backtest_cc(
            df, metal=metal,
            entryz=CONFIG['ENTRY_Z'], exitz=CONFIG['EXIT_Z'],
            stopz=CONFIG['STOP_Z'],   execcostbp=CONFIG['EXEC_COST_BP'],
        )
        cum_w = pnl.loc[s:e].cumsum()
        if len(cum_w) > 0:
            ax.plot(cum_w.index, cum_w.values, lw=1.5, color=color, label=lbl)
    ax.axhline(0, color='grey', lw=0.5, ls='--')
    ax.set_title(f'{win_name}\n{start}  →  {end}', fontsize=10)
    ax.set_ylabel('Cum P&L (bp)')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.grid(alpha=0.3)

plt.tight_layout()
out = CONFIG['OUT_DIR'] / 'cc_06_stress_test.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

stress_df.to_csv(CONFIG['OUT_DIR'] / 'cc_stress_results.csv', index=False)
print('Done. All outputs saved to outputs_comex/.')